# Selezione delle coppie 2: stesso settore e legame forte

**Perché un secondo notebook.** La prima selezione ([coppie.ipynb](../Old%20files/src/01_selezione_coppie/coppie.ipynb), ora in `Old files/`) ha scelto le coppie con la sola distanza dei prezzi. Qui si riparte da zero con un criterio più mirato: **stesso settore e legame forte**, con titoli scambiati. Il primo notebook resta com'è, come confronto.

**Cosa produce:** la lista delle coppie della **versione 2** (fino a 15). Il gruppo ha deciso di **tenerle tutte e 15**, anche se il programma ne prevede 6-9: la differenza va dichiarata nel report.

**I criteri del gruppo, e come diventano regole misurabili**

| Criterio | Regola in questo notebook | Dati usati |
|---|---|---|
| Stesso settore, legame economico riconoscibile | stessa **sotto-industria** GICS (più stretta del gruppo industriale usato prima) | `gics.csv` |
| Legame forte | correlazione dei rendimenti tra 0,6 e 0,95, poi classifica per **distanza** dei prezzi (SSD) | `daily` |
| Titoli realmente scambiati | valore scambiato giornaliero mediano ≥ 100 milioni di $ per *entrambi* i titoli | `daily` |
| *Non filtrano* | **costi** (indicazione del docente), **notizie** (altro passaggio), **eventi strutturali** (le coppie che li hanno restano, da segnalare nel report) | |

**Regola anti data-snooping (cosa facciamo e cosa non facciamo)**
1. **Distanze, correlazioni e liquidità giornaliera** usano solo dati giornalieri fino al **15 giugno 2026**.
2. **La lista iniziale la decide la regola**, applicata in modo meccanico dalla cella della parte 5: classifica per distanza, un titolo per coppia, tetto per settore. Nessuna coppia viene aggiunta, tolta o sostituita a mano.
3. **Le soglie sono fissate nella cella dei parametri** e non si ritoccano dopo aver guardato i risultati (Lo e MacKinlay, 1990; Harvey, Liu e Zhu, 2016).
4. **Da dichiarare nel report:** prima di scrivere questo notebook abbiamo guardato una prima volta i dati (distribuzione di liquidità dei titoli e un elenco provvisorio di coppie candidate), e il gruppo ha tolto in seguito i filtri su notizie, costi, eventi ed esclusione dei titoli già usati dopo aver visto le prime esecuzioni. Le soglie non sono ottimizzate sulla lista finale, ma non sono neppure indipendenti dai dati.
5. **Le diagnostiche della parte 6 non sono filtri**: descrivono le coppie, ma la lista la decide solo la regola.

## 1. Parametri, con il perché di ogni scelta

Legenda: **[ECO]** motivazione economica, **[ACC]** riferimento accademico, **[NOSTRA]** scelta nostra senza fonte: va dichiarata come tale.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, next(str(p) for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()))   # cartella del progetto: rende importabile src

import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import coint
from statsmodels.stats.multitest import multipletests

from src.config import GICS, PROCESSED, SELECTED
from src.data import load

# ---- Periodo di selezione ----
SEL_START = "2021-06-16"
SEL_END   = "2026-06-15"
# [ACC/ECO] Si finisce il 15 giugno 2026: la selezione usa solo dati giornalieri e non deve vedere il periodo successivo.
# [NOSTRA] 5 anni, come nella prima selezione, per poterle confrontare; si parte dal 2021 per lasciare fuori lo shock del Covid.

# ---- Filtri sul singolo titolo ----
MIN_DOLLAR_VOLUME = 100e6
# [ECO] Servono titoli dove si compra e si vende senza muovere il prezzo. Il doppio della soglia della prima selezione
#   (50 milioni) perche qui la liquidita e un criterio esplicito. [NOSTRA] la soglia: non esiste un valore ufficiale.
# Nessun filtro su notizie, costi ed eventi strutturali: scelta del gruppo (i costi non vanno considerati, indicazione del docente).
#
MAX_MISSING_PCT = 0.01
# [ECO] Dati completi: senza buchi la distanza tra le coppie non e falsata.

# ---- Filtri sulla coppia ----
MIN_CORR = 0.6
MAX_CORR = 0.95
# [ECO] Sotto 0,6 il legame e debole; sopra 0,95 i due titoli sono quasi la stessa azione (due classi di azioni della
#   stessa societa) e non c'e spread da sfruttare. Stesse soglie della prima selezione.
PAIR_LEVEL = "sub_industry"
# [ECO] Criterio 3: la sotto-industria GICS (es. "Semiconduttori") e piu stretta del gruppo industriale ("Semiconduttori e
#   apparecchiature") e rende il legame economico piu riconoscibile. Limite: GICS di oggi, non storico.
CAP_LEVEL = "industry_group"
MAX_PER_INDUSTRY = 3
# [NOSTRA] al massimo 3 coppie per gruppo industriale, per non concentrare tutto in un settore.
N_PAIRS = 15
# [NOSTRA] 15 coppie, tutte tenute per decisione del gruppo (il programma prevede 6-9: da dichiarare nel report).

## 2. Dati

- **Giornalieri** (fino al 15 giugno 2026): prezzo di chiusura aggiustato per split e dividendi (`PX_LAST`) e volume, per prezzi, rendimenti e valore scambiato.

In [2]:
daily = load("daily")
daily["ticker"] = daily["ticker"].astype(str)
daily = daily[(daily["date"] >= SEL_START) & (daily["date"] <= SEL_END)]
prices  = daily.pivot(index="date", columns="ticker", values="PX_LAST")
volume  = daily.pivot(index="date", columns="ticker", values="PX_VOLUME")
returns = daily.pivot(index="date", columns="ticker", values="ret")

years = len(prices) / 252
print(f"{len(prices)} giorni di borsa ({years:.1f} anni), {prices.shape[1]} ticker, dal {prices.index.min().date()} al {prices.index.max().date()}")
funnel = {"titoli di partenza": prices.shape[1]}

1255 giorni di borsa (5.0 anni), 501 ticker, dal 2021-06-16 al 2026-06-15


## 3. Filtri sul singolo titolo

Una coppia è tradabile solo se *entrambi* i titoli passano i filtri: liquidità giornaliera e dati completi. Ogni filtro riduce l'universo; l'imbuto si riporta nel report con i numeri.

**Cosa non si filtra, per scelta del gruppo:** i **costi** (indicazione del docente), le **notizie** (altro passaggio del programma) e gli **eventi strutturali** (fusioni, acquisizioni, scorpori): le coppie che li hanno restano in lista e vanno segnalate nel report.

In [3]:
def short(t): return t.split()[0]

dollar_volume = (prices * volume).median() / 1e6          # in milioni di $
missing = prices.isna().mean()
t = pd.DataFrame({"dv_m": dollar_volume, "missing": missing})
t["short"] = [short(x) for x in t.index]

f2 = t["dv_m"] < MIN_DOLLAR_VOLUME / 1e6
f4 = t["missing"] > MAX_MISSING_PCT

keep = ~(f2 | f4)
funnel["liquidi (valore scambiato giornaliero)"] = int((~f2).sum())
funnel["titoli ammessi (con pochi buchi)"] = int(keep.sum())
print(pd.Series(funnel, name="n").to_frame())

                                          n
titoli di partenza                      501
liquidi (valore scambiato giornaliero)  166
titoli ammessi (con pochi buchi)        154


## 4. Settore GICS e universo finale

`data/gics.csv` (da Bloomberg) dà `industry_group` e `sub_industry`. Per la coppia si chiede la **stessa sotto-industria**; per il tetto sul numero di coppie si usa il gruppo industriale. **Limite da dichiarare:** classificazione attuale, senza storia.

In [4]:
g = pd.read_csv(GICS).set_index("ticker")
univ = t[keep].index
px = prices[univ].ffill()
px = px.loc[:, px.notna().all()]                     # servono prezzi dal primo all'ultimo giorno
rets = returns[px.columns]
names = list(px.columns)
sub = pd.Series({n: g.loc[short(n), PAIR_LEVEL] if short(n) in g.index else None for n in names})
grp = pd.Series({n: g.loc[short(n), CAP_LEVEL] if short(n) in g.index else None for n in names})
print(f"{len(names)} titoli nell'universo finale; {sub.nunique()} sotto-industrie, di cui {(sub.value_counts() >= 2).sum()} con almeno 2 titoli")
funnel["con storia completa e GICS"] = int(sub.notna().sum())

154 titoli nell'universo finale; 64 sotto-industrie, di cui 38 con almeno 2 titoli


## 5. Selezione delle coppie: metodo della distanza, regola meccanica

**Idea** (Gatev, Goetzmann e Rouwenhorst, 2006): se due titoli sono stati "vicini" nel passato, probabilmente lo restano; quando si allontanano, si scommette sul ritorno. Come nella prima selezione: ogni prezzo si divide per il suo primo valore (1 $ investito all'inizio), la **SSD** è la somma dei quadrati delle differenze tra i due indici, e si ordina per SSD crescente.

**Cosa cambia rispetto alla prima volta:** i candidati sono solo le coppie della **stessa sotto-industria** tra i titoli che hanno già passato i filtri di liquidità e di dati completi. La classifica poi si percorre dall'alto: si prende una coppia se nessuno dei due titoli è già usato e se il suo gruppo industriale non ha già `MAX_PER_INDUSTRY` coppie. Si ferma a `N_PAIRS`. **Niente altro: nessuna coppia entra o esce a mano.**

In [5]:
norm = px / px.iloc[0]
x = norm.values.T
sq = (x ** 2).sum(axis=1)
SSD = sq[:, None] + sq[None, :] - 2 * x @ x.T
C = rets.corr().values

rows = []
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        if pd.notna(sub[names[i]]) and sub[names[i]] == sub[names[j]]:
            rows.append(dict(A=names[i], B=names[j], sotto_industria=sub[names[i]], gruppo=grp[names[i]], ssd=SSD[i, j], corr=C[i, j],
                             dv_min_m=min(t.loc[names[i], "dv_m"], t.loc[names[j], "dv_m"])))
same = pd.DataFrame(rows).sort_values("ssd").reset_index(drop=True)
same["rank_ssd"] = np.arange(1, len(same) + 1)
dist_funnel = {"coppie della stessa sotto-industria tra i titoli ammessi": len(same)}

identical = same[same["corr"] > MAX_CORR]
cand = same[(same["corr"] <= MAX_CORR) & (same["corr"] >= MIN_CORR)]
dist_funnel[f"correlazione tra {MIN_CORR} e {MAX_CORR}"] = len(cand)
print("Escluse come quasi identiche:"); display(identical[["A", "B", "corr", "ssd"]].round(3))
print("Scartate per correlazione bassa:", int((same["corr"] < MIN_CORR).sum()))

chosen, used, per_grp = [], set(), {}
for _, r in cand.iterrows():
    if r["A"] in used or r["B"] in used or per_grp.get(r["gruppo"], 0) >= MAX_PER_INDUSTRY:
        continue
    chosen.append(r); used.update([r["A"], r["B"]]); per_grp[r["gruppo"]] = per_grp.get(r["gruppo"], 0) + 1
    if len(chosen) == N_PAIRS:
        break
final = pd.DataFrame(chosen).reset_index(drop=True)
final.insert(0, "pair", final["A"].map(short) + "-" + final["B"].map(short))
dist_funnel[f"coppie finali (max {N_PAIRS})"] = len(final)
print(f"\nCoppie esaminate: {len(same)} (stessa sotto-industria); candidate dopo il filtro di correlazione: {len(cand)}.")
final[["pair", "sotto_industria", "rank_ssd", "ssd", "corr", "dv_min_m"]].round(2)

Escluse come quasi identiche:


,A,B,corr,ssd
1,GOOG UW Equity,GOOGL UW Equity,0.996,3.862


Scartate per correlazione bassa: 176

Coppie esaminate: 249 (stessa sotto-industria); candidate dopo il filtro di correlazione: 72.


,pair,sotto_industria,rank_ssd,ssd,corr,dv_min_m
0,MA-V,Servizi di elaborazione transa,1,3.85,0.88,319.43
1,CSX-UNP,Trasporto ferroviario,3,6.61,0.72,137.02
2,PYPL-XYZ,Servizi di elaborazione transa,4,8.74,0.61,119.01
3,HD-LOW,Vendita al dettaglio di prodot,5,10.39,0.87,169.11
4,DHR-TMO,Servizi e strumenti per scienz,6,15.92,0.81,165.60
5,CRM-WDAY,Software applicativi,7,18.26,0.65,154.31
6,JPM-WFC,Altre banche,8,20.09,0.76,261.05
7,NXPI-TXN,Semiconduttori,9,21.58,0.76,186.36
8,ELV-UNH,Cure mediche,10,22.84,0.63,137.38
9,CL-PG,Prodotti per la casa,17,33.17,0.72,109.75


**Come leggere la tabella.** `rank_ssd` è il posto per distanza tra le coppie della stessa sotto-industria; `dv_min_m` è il valore scambiato giornaliero (milioni di $) del titolo *peggiore* della coppia: una coppia vale quanto il suo anello più debole. Se le coppie sono meno di `N_PAIRS`, è un risultato, non un errore: lo scriviamo nel report e non allarghiamo i filtri per farne uscire di più.

## 6. Diagnostiche statistiche (non filtri)

Come nella prima selezione, qui **non si scartano coppie**: la lista è già fissata. Le misure servono a descrivere le coppie e a segnalare quelle più rischiose nel report; la selezione non si basa su di esse. Sul giornaliero e su 5 anni misurano orizzonti di giorni e mesi.

- **p-value di cointegrazione** (Engle-Granger sui log-prezzi, in entrambi i sensi, si tiene il peggiore): vicino a zero = lo spread tende a tornare verso la media.
- **Hedge ratio β**: regressione del log-prezzo di A su quello di B; vicino a 1 è coerente con capitale uguale sulle due gambe.
- **Half-life** (Ornstein-Uhlenbeck, Chan 2013): giorni per dimezzare uno scostamento.
- **Attraversamenti della media all'anno**: quante occasioni di trade offre lo spread.
- **Stabilità**: distanza e β nelle due metà del periodo.

In [6]:
logp = np.log(px)
half_a, half_b = px.index[: len(px) // 2], px.index[len(px) // 2 :]

def half_life(s):
    lam = np.polyfit(s[:-1], np.diff(s), 1)[0]
    return -np.log(2) / lam if lam < 0 else np.inf

def crossings(s):
    s = s - s.mean()
    return int((np.sign(s[1:]) != np.sign(s[:-1])).sum())

def worst_p(a, b, idx=None):
    u, v = (logp[a], logp[b]) if idx is None else (logp.loc[idx, a], logp.loc[idx, b])
    return max(coint(u, v)[1], coint(v, u)[1])

def ssd_on(a, b, idx):
    d = px.loc[idx, a] / px.loc[idx, a].iloc[0] - px.loc[idx, b] / px.loc[idx, b].iloc[0]
    return float((d ** 2).sum())

def beta_on(a, b, idx=None):
    u, v = (logp[a], logp[b]) if idx is None else (logp.loc[idx, a], logp.loc[idx, b])
    return np.polyfit(v, u, 1)[0]

diag = []
for _, r in final.iterrows():
    a, b = r["A"], r["B"]
    lr = np.log(px[a] / px[b]).values
    diag.append(dict(pair=r["pair"], spread_std_pct=lr.std() * 100, coint_p=worst_p(a, b), beta=beta_on(a, b), half_life_giorni=half_life(lr),
                     attraversamenti_anno=crossings(lr) / years, ssd_prima_meta=ssd_on(a, b, half_a), ssd_seconda_meta=ssd_on(a, b, half_b),
                     beta_prima=beta_on(a, b, half_a), beta_seconda=beta_on(a, b, half_b)))
diag = pd.DataFrame(diag)
diag.round(3)

,pair,spread_std_pct,coint_p,beta,half_life_giorni,attraversamenti_anno,ssd_prima_meta,ssd_seconda_meta,beta_prima,beta_seconda
0,MA-V,3.336,0.081,0.985,32.121,14.457,0.988,1.380,0.957,0.832
1,CSX-UNP,6.339,0.505,0.950,56.479,8.433,1.342,5.246,0.736,1.522
2,PYPL-XYZ,17.732,0.557,0.985,62.121,13.453,4.941,25.585,0.924,0.676
3,HD-LOW,5.152,0.144,0.975,52.404,7.630,4.818,3.785,0.872,0.740
4,DHR-TMO,7.056,0.286,0.907,61.593,10.642,6.982,4.543,0.937,0.867
5,CRM-WDAY,13.619,0.401,0.822,67.857,10.241,6.380,29.652,0.884,0.640
6,JPM-WFC,10.600,0.172,1.167,109.697,6.425,10.523,5.827,0.587,1.035
7,NXPI-TXN,11.296,0.790,0.885,66.262,10.441,4.846,42.680,1.008,0.504
8,ELV-UNH,11.288,0.114,0.645,60.335,10.843,2.315,12.200,0.928,0.598
9,CL-PG,6.059,0.065,1.139,62.264,8.835,21.404,3.346,0.303,1.092


**Test multipli.** La distanza classifica molte coppie, quindi alcune sono "vicine per caso". Il numero di coppie esaminate (parte 5) va riportato nel report.

## 7. Confronto con la prima selezione

Quali delle coppie scelte qui erano già nella prima selezione (`selected_pairs.csv`) e quali coppie della prima selezione non compaiono. Serve solo a documentare nel report quanto le due selezioni si sovrappongono.

In [7]:
OLD_SELECTED = PROCESSED / "selected_pairs.csv"    # lista della prima selezione: esiste solo se il suo notebook (in Old files) è stato eseguito
if not OLD_SELECTED.exists():
    print(f"manca {OLD_SELECTED.name}: il confronto con la prima selezione resta vuoto")
old = pd.read_csv(OLD_SELECTED) if OLD_SELECTED.exists() else pd.DataFrame({"pair": []})
cmp_new = final[["pair", "sotto_industria", "rank_ssd", "corr"]].copy()
cmp_new["nella_prima_selezione"] = cmp_new["pair"].isin(set(old["pair"]))
print("Coppie della prima selezione non presenti qui:", sorted(set(old["pair"]) - set(final["pair"])))
cmp_new.round(2)

Coppie della prima selezione non presenti qui: ['BAC-PNC', 'BNY-GS', 'ITW-OTIS', 'MCD-YUM', 'TFC-USB']


,pair,sotto_industria,rank_ssd,corr,nella_prima_selezione
0,MA-V,Servizi di elaborazione transa,1,0.88,True
1,CSX-UNP,Trasporto ferroviario,3,0.72,True
2,PYPL-XYZ,Servizi di elaborazione transa,4,0.61,True
3,HD-LOW,Vendita al dettaglio di prodot,5,0.87,True
4,DHR-TMO,Servizi e strumenti per scienz,6,0.81,False
5,CRM-WDAY,Software applicativi,7,0.65,False
6,JPM-WFC,Altre banche,8,0.76,False
7,NXPI-TXN,Semiconduttori,9,0.76,False
8,ELV-UNH,Cure mediche,10,0.63,False
9,CL-PG,Prodotti per la casa,17,0.72,False


## 8. Riepilogo per il report

**Imbuto dei titoli e delle coppie** (da riportare con i numeri):

In [8]:
display(pd.Series(funnel, name="n").to_frame())
pd.Series(dist_funnel, name="n").to_frame()

,n
titoli di partenza,501
liquidi (valore scambiato giornaliero),166
titoli ammessi (con pochi buchi),154
con storia completa e GICS,154


,n
coppie della stessa sotto-industria tra i titoli ammessi,249
correlazione tra 0.6 e 0.95,72
coppie finali (max 15),15


**Limiti da dichiarare nel report**
1. **Survivorship bias**: l'universo sono i titoli dell'S&P 500 di oggi.
2. **GICS attuale**, non storico.
3. **Soglie nostre** (liquidità giornaliera 100 milioni di $, correlazione 0,6-0,95, tetto 3 coppie per gruppo, 15 coppie): ragionevoli e dichiarate prima di vedere le coppie, ma non ottimizzate né derivate da una fonte. Alcune sono state scelte dopo aver guardato la distribuzione dei titoli (una prima esplorazione dei dati precede questo notebook).
4. **Eventi strutturali non esclusi**: la lista può contenere coppie con fusioni, acquisizioni o scorpori nel periodo (per scelta del gruppo). Per queste coppie il legame può essere cambiato e il rientro dello spread è meno affidabile: da segnalare coppia per coppia nel report.
5. **Costi non considerati** (come da indicazione del docente): nessun filtro su prezzo o costo di negoziazione. La liquidità è comunque un filtro (punto 3).
6. **Notizie**: nessun filtro né colonna in questo notebook; si valutano nel passaggio apposito del programma.
7. **Un solo periodo di selezione** di 5 anni: il legame può cambiare.
8. **Test multipli**: il numero di coppie esaminate è in parte 5.

## 9. Risultati e coppie scelte (testo per il report)

La cella seguente genera il testo dai risultati del notebook: se si rilancia con altri dati o parametri, numeri e tabella si aggiornano da soli. Le descrizioni economiche (`ECON`) e le note sugli eventi noti (`EVENTI`) sono scritte a mano dalla conoscenza generale e **vanno verificate prima di metterle nel report**.

In [9]:
from IPython.display import Markdown, display

# descrizione economica breve (testo scritto a mano, da verificare)
ECON = {
    "MA-V": "le due grandi reti di pagamento con carta: stesso modello di business, stessi clienti, ricavi legati alla spesa dei consumatori.",
    "CSX-UNP": "due ferrovie merci USA: stessi volumi di carico, stessa regolamentazione, stessi costi (carburante, lavoro).",
    "PYPL-XYZ": "pagamenti digitali per consumatori e commercianti (XYZ e Block): settore comune, ma ampiezza dello spread molto alta e correlazione bassa.",
    "HD-LOW": "i due grandi distributori USA per la casa e il fai-da-te: stessi clienti, stessa stagionalita, stessa sensibilita al mercato immobiliare.",
    "DHR-TMO": "strumenti e servizi per le scienze della vita: stessi clienti (laboratori, farmaceutiche, biotech) e stessi cicli di spesa.",
    "CRM-WDAY": "software aziendale in abbonamento (CRM e gestione del personale): stessi clienti e stessa sensibilita alla spesa IT.",
    "JPM-WFC": "due grandi banche universali USA: stessi fattori di tassi, credito e regolamentazione.",
    "NXPI-TXN": "semiconduttori analogici e di potenza: stessi mercati finali (auto, industria) e stesso ciclo.",
    "ELV-UNH": "assicurazioni sanitarie USA (managed care): stessi ricavi da premi, stessa regolamentazione e stesso costo delle cure.",
    "CL-PG": "beni di consumo per la casa e la persona: stessi canali di vendita (grande distribuzione) e stessi costi delle materie prime.",
    "GS-MS": "le due grandi banche d'investimento USA: stessi ricavi (trading, consulenza, gestione patrimoni).",
    "ISRG-SYK": "tecnologia medica per la chirurgia: stessi clienti (ospedali) e stessa spesa in conto capitale; business diversi (robot contro protesi).",
    "KO-PEP": "Coca-Cola e Pepsi: stesso mercato delle bevande analcoliche, stessi clienti e canali di vendita.",
    "BAC-C": "due grandi banche universali USA: stessi fattori di tassi e credito.",
    "COP-EOG": "produzione di petrolio e gas (esplorazione e produzione): ricavi legati agli stessi prezzi delle materie prime.",
}
# eventi strutturali noti nel periodo: NON sono filtri (scelta del gruppo), servono solo a segnalarli nel report. Da verificare.
EVENTI = {
    "CSX-UNP": "UNP ha annunciato la fusione con Norfolk Southern (2025)",
    "HD-LOW": "HD ha acquisito SRS Distribution (2024) e LOW FBM (2025)",
    "DHR-TMO": "DHR ha scorporato Veralto (2023)",
    "CRM-WDAY": "CRM ha acquisito Informatica (2025)",
    "JPM-WFC": "WFC: tetto agli attivi tolto dalla Fed (2025)",
    "BAC-C": "C: vendita di Banamex in corso",
    "COP-EOG": "COP ha acquisito Marathon Oil (2024)",
}

def fmt_hl(x): return "non definita" if not np.isfinite(x) else f"{x:.0f}"
res = final.merge(diag, on="pair")
res["evento"] = res["pair"].map(EVENTI).fillna("nessuno noto")
n_pairs = len(res)
n_old = int(cmp_new["nella_prima_selezione"].sum())
n_p05 = int((res["coint_p"] < 0.05).sum())
n_p10 = int((res["coint_p"] < 0.10).sum())
rows = ["| Coppia | Sotto-industria | Posto SSD (su %d) | Correlazione | Ampiezza spread | p-value coint. | Half-life (giorni) | Evento noto |" % len(same), "|---|---|---|---|---|---|---|---|"]
for _, r in res.iterrows():
    rows.append(f"| {r['pair']} | {r['sotto_industria']} | {r['rank_ssd']} | {r['corr']:.2f} | {r['spread_std_pct']:.1f}% | {r['coint_p']:.3f} | {fmt_hl(r['half_life_giorni'])} | {r['evento']} |")
desc = "\n".join(f"- **{p}**: {ECON.get(p, '(descrizione da scrivere)')}" for p in res["pair"])
text = "\n".join([
    "### Cosa abbiamo fatto",
    f"Dai {funnel['titoli di partenza']} titoli dell'indice abbiamo tenuto i {funnel['liquidi (valore scambiato giornaliero)']} con valore scambiato giornaliero mediano di almeno {MIN_DOLLAR_VOLUME/1e6:.0f} milioni di $ (titoli davvero scambiati) e i {funnel['titoli ammessi (con pochi buchi)']} con dati quasi completi nel periodo {SEL_START} - {SEL_END}. "
    f"Tra questi abbiamo preso solo coppie della **stessa sotto-industria GICS** ({dist_funnel['coppie della stessa sotto-industria tra i titoli ammessi']} coppie), quelle con correlazione dei rendimenti tra {MIN_CORR} e {MAX_CORR} ({len(cand)} coppie), e le abbiamo ordinate per **distanza** dei prezzi (SSD, Gatev et al., 2006). "
    f"Dall'alto abbiamo preso una coppia se nessun titolo era gia usato e se il gruppo industriale non aveva gia {MAX_PER_INDUSTRY} coppie, fino a {N_PAIRS}: ne sono uscite **{n_pairs}**.",
    "",
    "### Perche",
    "Stesso settore e legame forte sono le condizioni economiche perche lo spread torni alla media: aziende simili sono esposte agli stessi fattori. La distanza e il metodo di riferimento della letteratura sul pair trading e non richiede un modello statistico. La soglia di liquidita serve a tenere solo coppie realmente scambiate. Costi, notizie ed eventi strutturali non sono filtri: i primi per indicazione del docente, le notizie perche sono un altro passaggio del programma, gli eventi per scelta del gruppo (vanno solo segnalati).",
    "",
    "### Le coppie scelte (versione 2)",
    "\n".join(rows),
    "",
    desc,
    "",
    "### Cosa emerge",
    f"- {n_old} coppie su {n_pairs} erano gia nella versione 1 ({', '.join(cmp_new.loc[cmp_new['nella_prima_selezione'], 'pair'])}); le altre {n_pairs - n_old} sono nuove.",
    f"- Il legame statistico sul giornaliero e debole: solo {n_p05} " + ("coppia ha" if n_p05 == 1 else "coppie hanno") + f" p-value di cointegrazione sotto 0.05 ({n_p10} sotto 0.10). La lista e un elenco di candidate, non una prova: la prova arriva dai passaggi successivi.",
    "- Alcune coppie hanno half-life non definita (lo spread non rientra verso la media nel periodo) o distanza molto diversa tra le due meta del periodo: vedi tabella della parte 6. Non sono state scartate, per regola.",
    f"- Coppie esaminate: {len(same)} (test multipli: alcune sono vicine per caso).",
])
display(Markdown(text))

### Cosa abbiamo fatto
Dai 501 titoli dell'indice abbiamo tenuto i 166 con valore scambiato giornaliero mediano di almeno 100 milioni di $ (titoli davvero scambiati) e i 154 con dati quasi completi nel periodo 2021-06-16 - 2026-06-15. Tra questi abbiamo preso solo coppie della **stessa sotto-industria GICS** (249 coppie), quelle con correlazione dei rendimenti tra 0.6 e 0.95 (72 coppie), e le abbiamo ordinate per **distanza** dei prezzi (SSD, Gatev et al., 2006). Dall'alto abbiamo preso una coppia se nessun titolo era gia usato e se il gruppo industriale non aveva gia 3 coppie, fino a 15: ne sono uscite **15**.

### Perche
Stesso settore e legame forte sono le condizioni economiche perche lo spread torni alla media: aziende simili sono esposte agli stessi fattori. La distanza e il metodo di riferimento della letteratura sul pair trading e non richiede un modello statistico. La soglia di liquidita serve a tenere solo coppie realmente scambiate. Costi, notizie ed eventi strutturali non sono filtri: i primi per indicazione del docente, le notizie perche sono un altro passaggio del programma, gli eventi per scelta del gruppo (vanno solo segnalati).

### Le coppie scelte (versione 2)
| Coppia | Sotto-industria | Posto SSD (su 249) | Correlazione | Ampiezza spread | p-value coint. | Half-life (giorni) | Evento noto |
|---|---|---|---|---|---|---|---|
| MA-V | Servizi di elaborazione transa | 1 | 0.88 | 3.3% | 0.081 | 32 | nessuno noto |
| CSX-UNP | Trasporto ferroviario | 3 | 0.72 | 6.3% | 0.505 | 56 | UNP ha annunciato la fusione con Norfolk Southern (2025) |
| PYPL-XYZ | Servizi di elaborazione transa | 4 | 0.61 | 17.7% | 0.557 | 62 | nessuno noto |
| HD-LOW | Vendita al dettaglio di prodot | 5 | 0.87 | 5.2% | 0.144 | 52 | HD ha acquisito SRS Distribution (2024) e LOW FBM (2025) |
| DHR-TMO | Servizi e strumenti per scienz | 6 | 0.81 | 7.1% | 0.286 | 62 | DHR ha scorporato Veralto (2023) |
| CRM-WDAY | Software applicativi | 7 | 0.65 | 13.6% | 0.401 | 68 | CRM ha acquisito Informatica (2025) |
| JPM-WFC | Altre banche | 8 | 0.76 | 10.6% | 0.172 | 110 | WFC: tetto agli attivi tolto dalla Fed (2025) |
| NXPI-TXN | Semiconduttori | 9 | 0.76 | 11.3% | 0.790 | 66 | nessuno noto |
| ELV-UNH | Cure mediche | 10 | 0.63 | 11.3% | 0.114 | 60 | nessuno noto |
| CL-PG | Prodotti per la casa | 17 | 0.72 | 6.1% | 0.065 | 62 | nessuno noto |
| GS-MS | Banche di investimento e broke | 24 | 0.84 | 10.2% | 0.048 | 157 | nessuno noto |
| ISRG-SYK | Attrezzatura sanitaria | 25 | 0.63 | 14.2% | 0.084 | 110 | nessuno noto |
| KO-PEP | Bibite e bevande analcoliche | 27 | 0.68 | 15.4% | 0.946 | non definita | nessuno noto |
| BAC-C | Altre banche | 29 | 0.81 | 16.5% | 0.587 | non definita | C: vendita di Banamex in corso |
| COP-EOG | Esplorazione e produzione di p | 40 | 0.87 | 7.0% | 0.218 | 55 | COP ha acquisito Marathon Oil (2024) |

- **MA-V**: le due grandi reti di pagamento con carta: stesso modello di business, stessi clienti, ricavi legati alla spesa dei consumatori.
- **CSX-UNP**: due ferrovie merci USA: stessi volumi di carico, stessa regolamentazione, stessi costi (carburante, lavoro).
- **PYPL-XYZ**: pagamenti digitali per consumatori e commercianti (XYZ e Block): settore comune, ma ampiezza dello spread molto alta e correlazione bassa.
- **HD-LOW**: i due grandi distributori USA per la casa e il fai-da-te: stessi clienti, stessa stagionalita, stessa sensibilita al mercato immobiliare.
- **DHR-TMO**: strumenti e servizi per le scienze della vita: stessi clienti (laboratori, farmaceutiche, biotech) e stessi cicli di spesa.
- **CRM-WDAY**: software aziendale in abbonamento (CRM e gestione del personale): stessi clienti e stessa sensibilita alla spesa IT.
- **JPM-WFC**: due grandi banche universali USA: stessi fattori di tassi, credito e regolamentazione.
- **NXPI-TXN**: semiconduttori analogici e di potenza: stessi mercati finali (auto, industria) e stesso ciclo.
- **ELV-UNH**: assicurazioni sanitarie USA (managed care): stessi ricavi da premi, stessa regolamentazione e stesso costo delle cure.
- **CL-PG**: beni di consumo per la casa e la persona: stessi canali di vendita (grande distribuzione) e stessi costi delle materie prime.
- **GS-MS**: le due grandi banche d'investimento USA: stessi ricavi (trading, consulenza, gestione patrimoni).
- **ISRG-SYK**: tecnologia medica per la chirurgia: stessi clienti (ospedali) e stessa spesa in conto capitale; business diversi (robot contro protesi).
- **KO-PEP**: Coca-Cola e Pepsi: stesso mercato delle bevande analcoliche, stessi clienti e canali di vendita.
- **BAC-C**: due grandi banche universali USA: stessi fattori di tassi e credito.
- **COP-EOG**: produzione di petrolio e gas (esplorazione e produzione): ricavi legati agli stessi prezzi delle materie prime.

### Cosa emerge
- 5 coppie su 15 erano gia nella versione 1 (MA-V, CSX-UNP, PYPL-XYZ, HD-LOW, KO-PEP); le altre 10 sono nuove.
- Il legame statistico sul giornaliero e debole: solo 1 coppia ha p-value di cointegrazione sotto 0.05 (4 sotto 0.10). La lista e un elenco di candidate, non una prova: la prova arriva dai passaggi successivi.
- Alcune coppie hanno half-life non definita (lo spread non rientra verso la media nel periodo) o distanza molto diversa tra le due meta del periodo: vedi tabella della parte 6. Non sono state scartate, per regola.
- Coppie esaminate: 249 (test multipli: alcune sono vicine per caso).

## 10. Salvataggio

La lista della versione 2 (15 coppie) va in un file **nuovo** (`selected_pairs_2.csv`) per non sovrascrivere quella della versione 1.

In [10]:
out = final.merge(diag, on="pair")
out = out[["pair", "A", "B", "sotto_industria", "gruppo", "rank_ssd", "ssd", "corr", "dv_min_m", "coint_p", "beta", "half_life_giorni", "attraversamenti_anno"]]
out.to_csv(SELECTED, index=False)
out.round(3)

,pair,A,B,sotto_industria,gruppo,rank_ssd,ssd,corr,dv_min_m,coint_p,beta,half_life_giorni,attraversamenti_anno
0,MA-V,MA UN Equity,V UN Equity,Servizi di elaborazione transa,Servizi finanziari,1,3.854,0.880,319.430,0.081,0.985,32.121,14.457
1,CSX-UNP,CSX UW Equity,UNP UN Equity,Trasporto ferroviario,Trasporto,3,6.610,0.717,137.024,0.505,0.950,56.479,8.433
2,PYPL-XYZ,PYPL UW Equity,XYZ UN Equity,Servizi di elaborazione transa,Servizi finanziari,4,8.737,0.612,119.005,0.557,0.985,62.121,13.453
3,HD-LOW,HD UN Equity,LOW UN Equity,Vendita al dettaglio di prodot,Beni discrezionali Distribuzio,5,10.391,0.868,169.109,0.144,0.975,52.404,7.630
4,DHR-TMO,DHR UN Equity,TMO UN Equity,Servizi e strumenti per scienz,"Farmaceutica, Biotecnologie e",6,15.916,0.808,165.605,0.286,0.907,61.593,10.642
5,CRM-WDAY,CRM UN Equity,WDAY UW Equity,Software applicativi,Software e servizi,7,18.261,0.650,154.315,0.401,0.822,67.857,10.241
6,JPM-WFC,JPM UN Equity,WFC UN Equity,Altre banche,Banche,8,20.093,0.755,261.051,0.172,1.167,109.697,6.425
7,NXPI-TXN,NXPI UW Equity,TXN UW Equity,Semiconduttori,Semiconduttori e apparecchiatu,9,21.583,0.756,186.356,0.790,0.885,66.262,10.441
8,ELV-UNH,ELV UN Equity,UNH UN Equity,Cure mediche,Attrezzature e servizi sanitar,10,22.842,0.628,137.375,0.114,0.645,60.335,10.843
9,CL-PG,CL UN Equity,PG UN Equity,Prodotti per la casa,Prodotti per la casa e l'igien,17,33.171,0.720,109.750,0.065,1.139,62.264,8.835
